In [1]:
'''
Работа 1: регрессия средствами scikit-learn

Задание: 
A. Сгенерировать выборку данных, состоящую из элементов (трёхмерных точек) вида
    (x, y, x^2-y^2),
где x, y --- случайные числа из отрезка [0,1].
Б. Вывести эту выборку в виде трёхмерного облака точек

В. Провести регрессию на данной выборке методом линейной регрессии
    средствами библиотеки scikit-learn
В1. RANSAC вручную
Г. Вывести выборку и предсказания на один график 
   для визуальной оценки результатов

Д. Оценить ошибку данного метода

'''

# импортируем используемые библиотеки:
import numpy as np # базовая библиотека для численных методов
import sklearn.linear_model  # библиотека линейных моделей (в т.ч. линейной регрессии)
import matplotlib.pyplot as plt  # библиотека для вывода данных
from mpl_toolkits.mplot3d import axes3d, Axes3D  # дополнительные библиотеки
from sklearn.metrics import mean_squared_error  # для 3D визуализации


In [2]:
# А. Функция для генерации обучающих данных средствами numpy.random:
def generate_samples(NumSamples):
    v1 = np.random.uniform(0.1, 1, size=(NumSamples, 1))
    v2 = np.random.uniform(0.1, 1, size=(NumSamples, 1))
    epsilon = np.random.normal(loc=0, scale=2.1, size=(NumSamples, 1))
    res = np.power(v1, 2) - np.power(v2, 2) +epsilon
    v = np.concatenate((v1, v2), axis=1)
    # Объединение v и res в один массив points
    points = np.concatenate((v, res), axis=1)
    return (v, res, points)
points = generate_samples(5) 
print(points)

(array([[0.36423616, 0.42450306],
       [0.50733895, 0.17123883],
       [0.5525837 , 0.9622225 ],
       [0.61592363, 0.3161998 ],
       [0.67584693, 0.35869902]]), array([[-1.38815597],
       [-0.00562842],
       [ 1.04699369],
       [ 1.32201376],
       [-0.52632037]]), array([[ 0.36423616,  0.42450306, -1.38815597],
       [ 0.50733895,  0.17123883, -0.00562842],
       [ 0.5525837 ,  0.9622225 ,  1.04699369],
       [ 0.61592363,  0.3161998 ,  1.32201376],
       [ 0.67584693,  0.35869902, -0.52632037]]))


In [3]:
# Б. Функция для вывода выборки
def draw_samples(points):
    ''' v --- матрица независимых переменных,
    res --- вектор зависимой переменной '''
    fig = plt.figure(figsize = (8,8))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(points[:,0], points[:,1], points[:, 2], label='samples')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('x^2 + y^2')
    plt.show()


In [4]:
# Б1.
''' 
def ransacL(v, res, n_iterations=100, sample_size_ratio=0.6, t=0.1):
    n_samples = v.shape[0]
    sample_size = int(n_samples * sample_size_ratio)
    best_model = None
    best_inlier_num = 0

    model = sklearn.linear_model.LinearRegression()
    for _ in range(n_iterations):
        # Выбор случайных образцов
        sample = np.random.choice(n_samples, sample_size, replace=False)
        #X_sample = v[sample]
        #y_sample = y[sample]
        #v = numpy.concatenate((X_sample, y_sample), axis=1)
        v_sample = v[sample]
        res_sample = res[sample]
        # Подгонка модели к выбранным образцам
        model_sample = model.fit(v_sample, res_sample)
        res_sample = model_sample.predict(v)

        # Нахождение внутренних точек (inliers)
        residuals = np.abs(res_sample - res)
        inliers = residuals < t
        inlier_num = np.count_nonzero(inliers)

        # Обновление лучшей модели, если текущая модель лучше
        if inlier_num > best_inlier_num:
            best_inlier_num = inlier_num
            best_model = model_sample

    return best_model.predict
'''

' \ndef ransacL(v, res, n_iterations=100, sample_size_ratio=0.6, t=0.1):\n    n_samples = v.shape[0]\n    sample_size = int(n_samples * sample_size_ratio)\n    best_model = None\n    best_inlier_num = 0\n\n    model = sklearn.linear_model.LinearRegression()\n    for _ in range(n_iterations):\n        # Выбор случайных образцов\n        sample = np.random.choice(n_samples, sample_size, replace=False)\n        #X_sample = v[sample]\n        #y_sample = y[sample]\n        #v = numpy.concatenate((X_sample, y_sample), axis=1)\n        v_sample = v[sample]\n        res_sample = res[sample]\n        # Подгонка модели к выбранным образцам\n        model_sample = model.fit(v_sample, res_sample)\n        res_sample = model_sample.predict(v)\n\n        # Нахождение внутренних точек (inliers)\n        residuals = np.abs(res_sample - res)\n        inliers = residuals < t\n        inlier_num = np.count_nonzero(inliers)\n\n        # Обновление лучшей модели, если текущая модель лучше\n        if inli

In [5]:
def fit_plane(points):
    
    if points.shape[1] != 3:
        raise ValueError("points должен быть массивом размером (N, 3)")

    # Создание матрицы A
    A = np.hstack([points, np.ones((points.shape[0], 1))])

    # Создание матрицы B
    B = np.zeros((points.shape[0], 1))

    # Решение системы уравнений методом наименьших квадратов
    C, _, _, _ = np.linalg.lstsq(A, B, rcond=None)

    # Параметры плоскости
    A, B, C, D = C[0], C[1], C[2], C[3]

    return A, B, C, D

def distance_to_plane(plane, points):
    """Вычисление расстояний от точек до плоскости Ax + By + Cz + D = 0."""
    A, B, C, D = plane
    numerator = np.abs(A * points[:, 0] + B * points[:, 1] + C * points[:, 2] + D)
    denominator = np.sqrt((A+0.001)**2 + B**2 + C**2)

    # Проверка на деление на ноль
    if denominator == 0:
        raise ValueError("Знаменатель равен нулю, что делает деление невозможным.")

    distances = numerator / denominator
    return distances
def predictor_ransac(points, n_iterations=100, threshold=0.01, min_inliers=0.5):
    n_points = len(points)
    best_model = None
    best_inlier_num = 0
    for _ in range(n_iterations):
        # Случайный выбор 3 точек
        sample_indices = np.random.choice(n_points, 3, replace=False)
        sample_points = points[sample_indices]
        # Подгонка плоскости
        plane = fit_plane(sample_points)

        # Расчет расстояний до плоскости
        distances = distance_to_plane(plane, points)

        # Определение инлайнеров
        inliers = distances < threshold
        inliers_count = np.sum(inliers)

        best_inliers_count = 0
        best_plane = None
        # Проверка на лучшее количество инлайнеров
        if inliers_count > best_inliers_count:
            best_inliers_count = inliers_count
            best_plane = plane

    return best_plane, best_inliers_count

In [6]:
def calculate_ransac_by_standard_method(v, res):
    model = sklearn.linear_model.RANSACRegressor(random_state=1);
    model.fit(v, res)
    predictor = model.predict
    return predictor

In [7]:
# В. Функция для вывода 
def calculate_regression_by_standard_method(v, res):
    # создаём объект модели, передавая в него параметры регуляризации
    model = sklearn.linear_model.LinearRegression()
    # обучаем модель на выборке встроенным в неё методом fit
    model.fit(v, res)
    # получаем предсказания для выборки в виде вектора
    # того же формата, что и res
    #predictor = lambda v: model.predict(v)
    # возвращаем предсказания
    return model.predict

In [8]:
# Г. Функция для вывода выборки вместе с предсказаниями
def draw_samples_and_prediction(v, res, prediction):
    ''' v --- матрица независимых переменных,
    res --- вектор зависимой переменной,
    prediction --- вектор значений этой зависимой переменной,
       предсказанной по v '''
    fig = plt.figure(figsize = (8,8))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(v[:,0], v[:,1], res, label='samples', marker='o')
    ax.scatter(v[:,0], v[:,1], prediction, label='prediction', marker='^')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('x^2+y^2')
    plt.show()

In [9]:
# Д. Функция для расчёта ошибки предсказания (среднего отклонения
#    предсказанного значения функции от реального).
def calc_error(res, prediction):
    '''
    res и prediction --- векторы-столбцы одного размера
    '''
    return np.linalg.norm(res - prediction) / res.shape[0]


In [10]:
# сгенерируем выборку в виде двух матриц, чтобы предсказываемое значение
# находилось в отдельной переменной

# зададим фиксированное начальное состояние генератора случайных чисел,
# чтобы можно было воспроизвести результаты (т.е. в выборку при каждом 
# запуске будут входить одни и те же псевдослучайные числа).
#numpy.random.seed(17001)

# создаём выборку
v, res, points = generate_samples(1000)
v1, res1, points = generate_samples(1000)
# выводим её для визуальной оценки
print("Sampels")
draw_samples(v, res)
draw_samples(v1, res1)
# обучаем модель
predictor_lin_reg = calculate_regression_by_standard_method(v, res)
predictor_ransac = ransac(v, res)
predictor_ransac_std = calculate_ransac_by_standard_method(v, res)
# выводим предсказания
print("Predictions")
draw_samples(v1, predictor_lin_reg(v1))
draw_samples(v1, predictor_ransac(v1))
draw_samples(v1, predictor_ransac_std(v1))


# выводим одновременно предсказания и исходные данные
'''
print("Predictions and samples")
draw_samples_and_prediction(v, res, predictor_lin_reg(v))
draw_samples_and_prediction(v1, res1, predictor_lin_reg(v1))
draw_samples_and_prediction(v1, res1, predictor_ransac(v1))
'''
# рассчитываем ошибку предсказания
error = calc_error(res, predictor_lin_reg(v))
error1 = calc_error(res1, predictor_lin_reg(v1))
error2 = calc_error(res1, predictor_ransac(v1))
err3 = calc_error(res1, predictor_ransac_std(v1))
print('MSE is {:.3f}'.format(error))
print('MSE is {:.3f}'.format(error1))
print('MSE_RANSAC is {:.3f}'.format(error2))
print('MSE_RANSAC STD is {:.3f}'.format(err3))

Sampels


TypeError: draw_samples() takes 1 positional argument but 2 were given